In [ ]:
# 1. 必要なツールのインストール
!pip install -q bitsandbytes accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-7B-Instruct"

# 4bit量子化の設定（VRAMを節約）
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Tokenizer ロード中...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("7Bモデル（4bit）ロード中...（3分程度）")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("ロード完了！")

In [ ]:
#Google公式SDKをインストール
!pip install -q google-genai

In [ ]:
# モデルがメモリ上に存在して動くかテスト
inputs = tokenizer("おい", return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=10)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# memories.json を空リストでリセット
with open("memories.json", "w", encoding="utf-8") as f:
  f.write("[]")

In [ ]:
import os

# 必ず /content を基準にする
%cd /content

REPO_URL = "https://github.com/Towa-1103/Experiment.git"
TARGET_DIR = "/content/Experiment"

if not os.path.exists(TARGET_DIR):
  !git clone {REPO_URL}
  %cd {TARGET_DIR}
else:
  %cd {TARGET_DIR}
  !git pull

print("\n 現在地:", os.getcwd())
print("\n リポジトリの同期が完了しました。")

In [ ]:
!git pull
import glob
import importlib
import os
import time
import dialogue_processor
import memory_manager
from google.colab import userdata

importlib.reload(dialogue_processor)
importlib.reload(memory_manager)

from dialogue_processor import extract_memories_with_gemini
from memory_manager import MemoryManager

# --- 設定 ---
GEMINI_API_KEY = userdata.get("Gemini_API_Key")
LOG_DIR = "./LINE_talk"
MEMORY_FILE = "memories.json"

# クリーンな状態でリセット
with open(MEMORY_FILE, "w", encoding="utf-8") as f:
  f.write("[]")

manager = MemoryManager(MEMORY_FILE)
target_files = sorted(glob.glob(os.path.join(LOG_DIR, "*.txt")))
print(f"処理対象: {len(target_files)} 件\n")

for idx, file_path in enumerate(target_files, 1):
  file_name = os.path.basename(file_path)

  # ファイル名から相手名を推定（例: "Friend_A_251125.txt" -> "Friend_A"）
  base_name = os.path.splitext(file_name)[0]
  inferred_sender = (
      base_name.rsplit("_", 1)[0] if "_" in base_name else base_name
  )

  print(
      f"[{idx}/{len(target_files)}] 処理中: {file_name} (相手推定: {inferred_sender})"
  )

  with open(file_path, "r", encoding="utf-8") as f:
    raw_log = f.read()

  if not raw_log.strip():
    continue

  extracted = extract_memories_with_gemini(
      raw_log, GEMINI_API_KEY, default_sender=inferred_sender, min_importance=3
  )
  added = manager.add_memories(extracted)
  print(f"  -> 抽出: {len(extracted)} 件 / 新規保存: {added} 件")

  # 無料枠制限の待機（15回/分）
  time.sleep(4)

print(f"\n全件完了！ 最終記憶数: {len(manager.get_all())} 件")

In [ ]:
# 直前の extracted の中身を表示
for i, m in enumerate(extracted, 1):
  print(f"[{i}] 日時: {m.get('timestamp')} | 要約: {m.get('text')[:30]}...")

In [ ]:
# logs.zip を Experiment フォルダ配下に解凍する場合
!unzip -q /content/Experiment/LINE_talk.zip -d /content/Experiment/